In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

audio_dir = "C:/Users/zhaoh/Desktop/processed_train/audio"
metadata_file = "C:/Users/zhaoh/Desktop/processed_train/metadata_extracted.csv"

metadata = pd.read_csv(metadata_file)

In [4]:
def extract_mfcc(file_path):
    try:
        # Focusing on the significant 0.5s to 3.5s window
        audio, sr = librosa.load(file_path, offset=0.5, duration=3.0)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40, n_fft=512, hop_length=256)
        
        # Ensure consistent width (130 frames for 3s)
        if mfccs.shape[1] < 130:
            mfccs = np.pad(mfccs, ((0, 0), (0, 130 - mfccs.shape[1])), mode='constant')
        else:
            mfccs = mfccs[:, :130]
        return mfccs
    except Exception as e:
        return None

In [5]:
# From audio_data_learning.ipynb

# ==========================================
# 3. Process Data
# ==========================================
features, labels = [], []
print("Extracting features... this may take a moment.")

for _, row in metadata.iterrows():
    # Use your defined audio_dir path
    path = row['audio_path']
    
    feat = extract_mfcc(path)
    if feat is not None:
        features.append(feat)
        labels.append(row['emotion'])
# Safety check to avoid ValueError: n_samples=0
if len(features) == 0:
    raise ValueError(f"No audio files found or loaded at: {audio_dir}. Please check if the folder exists and contains .wav files.")

X = np.array(features)
le = LabelEncoder()
y = le.fit_transform(labels)
num_classes = len(le.classes_)

# ==========================================
# 4. Manual Class Weights (Handles Neutral imbalance)
# ==========================================
counts = np.bincount(y)
# Formula: Total / (Num_Classes * Count_per_Class)
weights = len(y) / (num_classes * counts)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

print(f"Classes: {le.classes_}")
print(f"Counts: {counts}")
print(f"Weights applied: {weights.round(2)}")

# ==========================================
# 5. Stratified 80/10/10 Split
# ==========================================
# Split 80% Train, 20% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Split remaining 20% into 10% Eval and 10% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

def create_loader(X_data, y_data, batch_size=64):
    # CNN input: (Batch, Channels, MFCCs, Time)
    X_t = torch.FloatTensor(X_data).unsqueeze(1).to(device)
    y_t = torch.LongTensor(y_data).to(device)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

train_loader = create_loader(X_train, y_train)
val_loader = create_loader(X_val, y_val)
test_loader = create_loader(X_test, y_test)

# ==========================================
# 6. CNN Model Architecture
# ==========================================
class EmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super(EmotionCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 10 * 32, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.conv_block(x))

model = EmotionCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 7. Training Loop (50 Epochs, Batch 64)
# ==========================================
print(f"\nStarting training on {len(X_train)} samples...")

for epoch in range(50):
    model.train()
    correct, total = 0, 0
    
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

    # Display Accuracy Every 5 Epochs
    if (epoch + 1) % 5 == 0:
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for v_in, v_tar in val_loader:
                v_out = model(v_in)
                _, v_pred = torch.max(v_out.data, 1)
                v_total += v_tar.size(0)
                v_correct += (v_pred == v_tar).sum().item()
        
        train_acc = 100 * correct / total
        val_acc = 100 * v_correct / v_total
        print(f"Epoch [{epoch+1:02d}/50] | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# ==========================================
# 8. Final Test
# ==========================================
model.eval()
t_correct, t_total = 0, 0
with torch.no_grad():
    for t_in, t_tar in test_loader:
        t_out = model(t_in)
        _, t_pred = torch.max(t_out.data, 1)
        t_total += t_tar.size(0)
        t_correct += (t_pred == t_tar).sum().item()

print(f"\nFinal Test Accuracy: {100 * t_correct / t_total:.2f}%")

Extracting features... this may take a moment.


C:\Users\zhaoh\AppData\Local\Temp\ipykernel_14256\170839604.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, offset=0.5, duration=3.0)
c:\Users\zhaoh\Desktop\CDS\venv\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\zhaoh\AppData\Local\Temp\ipykernel_14256\170839604.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, offset=0.5, duration=3.0)
c:\Users\zhaoh\Desktop\CDS\venv\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
c:\Users\zhaoh\Desktop\CDS\venv\lib\site-packages\librosa\core\s

Classes: ['anger' 'disgust' 'fear' 'joy' 'neutral' 'sadness' 'surprise']
Counts: [ 757  174  170 1161 3123  490  805]
Weights applied: [1.26 5.48 5.61 0.82 0.31 1.95 1.19]


AttributeError: module 'sympy' has no attribute 'printing'

In [ ]:
from sklearn.preprocessing import StandardScaler

# ==========================================
# 3. Feature Scaling (Crucial for convergence)
# ==========================================
# Flatten X to scale it, then reshape back to (Samples, 40, 130)
X_shape = X.shape
X_flat = X.reshape(X_shape[0], -1)
scaler = StandardScaler()
X = scaler.fit_transform(X_flat).reshape(X_shape)
print("Features scaled successfully.")

# ==========================================
# 4. Handle Imbalance & Split (80/10/10)
# ==========================================
counts = np.bincount(y)
weights = len(y) / (num_classes * counts)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

def create_loader(X_data, y_data, batch_size=32):
    # shuffle=True ensures mini-batch randomization every epoch
    X_t = torch.FloatTensor(X_data).unsqueeze(1).to(device)
    y_t = torch.LongTensor(y_data).to(device)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

train_loader = create_loader(X_train, y_train)
val_loader = create_loader(X_val, y_val)
test_loader = create_loader(X_test, y_test)

# ==========================================
# 5. CNN Architecture
# ==========================================
class EmotionDeepCNN(nn.Module):
    def __init__(self, num_classes):
        super(EmotionDeepCNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 5 * 16, 512), # Increased dense layer size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.fc(self.layer3(self.layer2(self.layer1(x))))

model = EmotionDeepCNN(num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
# Scheduler to lower LR if learning stalls
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# ==========================================
# 6. Training Loop (100 Epochs)
# ==========================================
print(f"Starting training for 100 epochs...")
for epoch in range(100):
    model.train()
    correct, total = 0, 0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        _, pred = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (pred == targets).sum().item()

    # Evaluation
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for v_in, v_tar in val_loader:
            v_out = model(v_in)
            _, v_pred = torch.max(v_out.data, 1)
            v_total += v_tar.size(0)
            v_correct += (v_pred == v_tar).sum().item()
    
    val_acc = 100 * v_correct / v_total
    scheduler.step(val_acc)

    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        train_acc = 100 * correct / total
        print(f"Epoch {epoch+1:03d}/100 | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# ==========================================
# 7. Final Test Accuracy
# ==========================================
model.eval()
t_correct, t_total = 0, 0
with torch.no_grad():
    for t_in, t_tar in test_loader:
        t_out = model(t_in)
        _, t_pred = torch.max(t_out.data, 1)
        t_total += t_tar.size(0)
        t_correct += (t_pred == t_tar).sum().item()

print(f"\nFinal Test Accuracy: {100 * t_correct / t_total:.2f}%")

In [ ]:
# ==========================================
# 4. Regularized CNN Architecture
# ==========================================
class RegularizedEmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super(RegularizedEmotionCNN, self).__init__()
        
        # Block 1
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2) # Added dropout early
        )
        
        # Block 2
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3) # Increased dropout
        )
        
        # Block 3
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.4) # Stronger dropout
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 5 * 16, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512), # Added Batch Norm to fully connected layer
            nn.Dropout(0.5),     # Strongest dropout before output
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.fc(self.layer3(self.layer2(self.layer1(x))))

model = RegularizedEmotionCNN(num_classes).to(device)

# ==========================================
# 5. Regularized Loss & Optimizer
# ==========================================
# optimizer: increased weight_decay to 0.05 for stronger L2 regularization
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)

# scheduler: aggressive LR reduction to converge smoothly
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

# Label Smoothing: Reduces overconfidence (the 0.1 value is standard)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

# ==========================================
# 6. Training Loop (100 Epochs)
# ==========================================
for epoch in range(100):
    model.train()
    correct, total = 0, 0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        _, pred = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (pred == targets).sum().item()

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for v_in, v_tar in val_loader:
            v_out = model(v_in)
            _, v_pred = torch.max(v_out.data, 1)
            v_total += v_tar.size(0)
            v_correct += (v_pred == v_tar).sum().item()
    
    val_acc = 100 * v_correct / v_total
    scheduler.step(val_acc)

    if (epoch + 1) % 5 == 0:
        train_acc = 100 * correct / total
        print(f"Epoch {epoch+1:03d}/100 | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")

# ==========================================
# 7. Final Test
# ==========================================
model.eval()
t_correct, t_total = 0, 0
with torch.no_grad():
    for t_in, t_tar in test_loader:
        t_out = model(t_in)
        _, t_pred = torch.max(t_out.data, 1)
        t_total += t_tar.size(0)
        t_correct += (t_pred == t_tar).sum().item()

print(f"\nFinal Test Accuracy: {100 * t_correct / t_total:.2f}%")

In [11]:
# ==========================================
# 3. Handle Imbalance & Split (80/10/10)
# ==========================================
counts = np.bincount(y)
weights = len(y) / (num_classes * counts)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

def create_loader(X_data, y_data, batch_size=32, shuffle=True):
    X_t = torch.FloatTensor(X_data).unsqueeze(1).to(device)
    y_t = torch.LongTensor(y_data).to(device)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = create_loader(X_train, y_train, shuffle=True)
val_loader = create_loader(X_val, y_val, shuffle=False)
test_loader = create_loader(X_test, y_test, shuffle=False)

# ==========================================
# 4. Regularized CNN Architecture
# ==========================================
class RegularizedEmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2)
        )
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )
        
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.4)
        )
        
        # Placeholder for fc layer, will initialize later
        self.fc = None
        self.num_classes = num_classes

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        # Flatten
        x = torch.flatten(x, 1)
        
        # Initialize fc dynamically if not done yet
        if self.fc is None:
            self.fc = nn.Sequential(
                nn.Linear(x.shape[1], 512),
                nn.ReLU(),
                nn.BatchNorm1d(512),
                nn.Dropout(0.5),
                nn.Linear(512, self.num_classes)
            ).to(x.device)
        
        return self.fc(x)

model = RegularizedEmotionCNN(num_classes).to(device)

# ==========================================
# 5. Regularized Loss & Optimizer
# ==========================================
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.001)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

# ==========================================
# 6. Training Loop (100 Epochs)
# ==========================================
for epoch in range(100):
    model.train()
    correct, total = 0, 0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        _, pred = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (pred == targets).sum().item()

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for v_in, v_tar in val_loader:
            v_out = model(v_in)
            _, v_pred = torch.max(v_out.data, 1)
            v_total += v_tar.size(0)
            v_correct += (v_pred == v_tar).sum().item()
    
    val_acc = 100 * v_correct / v_total
    # scheduler.step(val_acc)

    if (epoch + 1) % 5 == 0:
        train_acc = 100 * correct / total
        print(f"Epoch {epoch+1:03d}/100 | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")

# ==========================================
# 7. Final Test
# ==========================================
model.eval()
t_correct, t_total = 0, 0
with torch.no_grad():
    for t_in, t_tar in test_loader:
        t_out = model(t_in)
        _, t_pred = torch.max(t_out.data, 1)
        t_total += t_tar.size(0)
        t_correct += (t_pred == t_tar).sum().item()

print(f"\nFinal Test Accuracy: {100 * t_correct / t_total:.2f}%\nchange: use full audio length")

Epoch 005/100 | Train Acc: 17.18% | Val Acc: 14.82% | LR: 0.010000
Epoch 010/100 | Train Acc: 17.65% | Val Acc: 22.75% | LR: 0.010000
Epoch 015/100 | Train Acc: 18.54% | Val Acc: 20.96% | LR: 0.010000
Epoch 020/100 | Train Acc: 18.62% | Val Acc: 19.16% | LR: 0.010000
Epoch 025/100 | Train Acc: 19.12% | Val Acc: 19.46% | LR: 0.010000
Epoch 030/100 | Train Acc: 18.45% | Val Acc: 20.06% | LR: 0.010000
Epoch 035/100 | Train Acc: 19.16% | Val Acc: 18.71% | LR: 0.010000
Epoch 040/100 | Train Acc: 20.71% | Val Acc: 20.36% | LR: 0.010000
Epoch 045/100 | Train Acc: 22.38% | Val Acc: 18.86% | LR: 0.010000
Epoch 050/100 | Train Acc: 23.07% | Val Acc: 17.51% | LR: 0.010000
Epoch 055/100 | Train Acc: 24.64% | Val Acc: 21.26% | LR: 0.010000
Epoch 060/100 | Train Acc: 24.57% | Val Acc: 19.16% | LR: 0.010000
Epoch 065/100 | Train Acc: 26.91% | Val Acc: 19.91% | LR: 0.010000
Epoch 070/100 | Train Acc: 27.45% | Val Acc: 17.96% | LR: 0.010000
Epoch 075/100 | Train Acc: 29.04% | Val Acc: 19.61% | LR: 0.01

In [12]:
import copy

class FullAudioEmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super(FullAudioEmotionCNN, self).__init__()
        
        self.conv_blocks = nn.Sequential(
            # Layer 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            # Layer 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.3),
            # Layer 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.4)
        )
        
        self.classifier = nn.Sequential(
            # Collapses (256, 5, 27) -> (256, 1, 1)
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_blocks(x)
        return self.classifier(x)

model = FullAudioEmotionCNN(num_classes).to(device)

# ==========================================
# 5. Optimization & Checkpointing
# ==========================================
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)
counts = np.bincount(y)
class_weights = torch.tensor(len(y)/(num_classes*counts), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

best_val_acc = 0.0
best_model_wts = None

# ==========================================
# 6. Training Loop
# ==========================================
for epoch in range(100):
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for v_in, v_tar in val_loader:
            outputs = model(v_in)
            _, pred = torch.max(outputs.data, 1)
            v_total += v_tar.size(0)
            v_correct += (pred == v_tar).sum().item()
    
    val_acc = 100 * v_correct / v_total
    scheduler.step(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:03d}/100 | Val Acc: {val_acc:.2f}% | Best: {best_val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")

# ==========================================
# 7. Final Test (Load Best Model)
# ==========================================
model.load_state_dict(best_model_wts)
model.eval()
t_correct, t_total = 0, 0
with torch.no_grad():
    for t_in, t_tar in test_loader:
        t_out = model(t_in)
        _, t_pred = torch.max(t_out.data, 1)
        t_total += t_tar.size(0)
        t_correct += (t_pred == t_tar).sum().item()

print(f"\nFinal Test Accuracy (Best Model): {100 * t_correct / t_total:.2f}%")

Epoch 005/100 | Val Acc: 6.44% | Best: 15.27% | LR: 0.001000
Epoch 010/100 | Val Acc: 5.84% | Best: 27.84% | LR: 0.001000
Epoch 015/100 | Val Acc: 7.93% | Best: 27.84% | LR: 0.001000
Epoch 020/100 | Val Acc: 7.49% | Best: 27.84% | LR: 0.000500
Epoch 025/100 | Val Acc: 10.18% | Best: 27.84% | LR: 0.000250
Epoch 030/100 | Val Acc: 6.89% | Best: 27.84% | LR: 0.000250
Epoch 035/100 | Val Acc: 3.29% | Best: 27.84% | LR: 0.000125
Epoch 040/100 | Val Acc: 2.69% | Best: 27.84% | LR: 0.000125
Epoch 045/100 | Val Acc: 6.14% | Best: 27.84% | LR: 0.000063
Epoch 050/100 | Val Acc: 7.04% | Best: 27.84% | LR: 0.000031
Epoch 055/100 | Val Acc: 6.59% | Best: 27.84% | LR: 0.000031
Epoch 060/100 | Val Acc: 6.59% | Best: 27.84% | LR: 0.000016
Epoch 065/100 | Val Acc: 6.74% | Best: 27.84% | LR: 0.000008
Epoch 070/100 | Val Acc: 7.19% | Best: 27.84% | LR: 0.000008
Epoch 075/100 | Val Acc: 6.14% | Best: 27.84% | LR: 0.000004
Epoch 080/100 | Val Acc: 6.14% | Best: 27.84% | LR: 0.000004
Epoch 085/100 | Val Acc

In [15]:
# ==========================================
# 2. Feature Extraction (3-Channel MFCC)
# ==========================================
def extract_mfcc(file_path):
    try:
        audio, sr = librosa.load(file_path, offset=0.5)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        
        # CHANGE: Added Delta and Delta-Delta features
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        
        # Stack into 3 channels (like an image)
        features = np.stack([mfcc, delta, delta2]) # Shape: (3, 40, Width)
        
        max_width = 216
        if features.shape[2] < max_width:
            features = np.pad(features, ((0,0), (0,0), (0, max_width - features.shape[2])), mode='constant')
        else:
            features = features[:, :, :max_width]
        return features
    except: return None

# ==========================================
# 3. Data Loaders
# ==========================================
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

def create_loader(X_data, y_data, shuffle=True):
    X_t = torch.FloatTensor(X_data).to(device) # Already has channel dim
    y_t = torch.LongTensor(y_data).to(device)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=shuffle)

train_loader = create_loader(X_train, y_train)
val_loader = create_loader(X_val, y_val, shuffle=False)
test_loader = create_loader(X_test, y_test, shuffle=False)

# ==========================================
# 4. Architecture (3-Channel Input)
# ==========================================
class TripleChannelEmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # CHANGE: Input channels changed from 1 to 3
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.3),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.4)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.conv_blocks(x))

model = TripleChannelEmotionCNN(num_classes).to(device)

# ==========================================
# 5. Optimization
# ==========================================
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)
counts = np.bincount(y)
class_weights = torch.tensor(len(y)/(num_classes*counts), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

best_val_acc = 0.0
best_model_wts = None

# ==========================================
# 6. Training Loop (with Training Acc)
# ==========================================
for epoch in range(100):
    model.train()
    t_correct, t_total = 0, 0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        _, pred = torch.max(outputs, 1)
        t_total += targets.size(0); t_correct += (pred == targets).sum().item()

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for v_in, v_tar in val_loader:
            v_out = model(v_in)
            _, v_pred = torch.max(v_out, 1)
            v_total += v_tar.size(0); v_correct += (v_pred == v_tar).sum().item()
    
    train_acc = 100 * t_correct / t_total
    val_acc = 100 * v_correct / v_total
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:03d}/100 | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Best: {best_val_acc:.2f}%")

# ==========================================
# 7. Final Test
# ==========================================
model.load_state_dict(best_model_wts)
model.eval()
f_correct, f_total = 0, 0
with torch.no_grad():
    for f_in, f_tar in test_loader:
        f_out = model(f_in)
        _, f_pred = torch.max(f_out, 1)
        f_total += f_tar.size(0); f_correct += (f_pred == f_tar).sum().item()

print(f"\nFinal Test Accuracy: {100 * f_correct / f_total:.2f}%")

RuntimeError: Given groups=1, weight of size [64, 3, 3, 3], expected input[1, 32, 40, 130] to have 3 channels, but got 32 channels instead

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

def evaluate_model(model, loader, encoder):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())
    
    # 1. Classification Report (Precision, Recall, F1)
    # This lists EVERY emotion and how the model handled it
    report = classification_report(all_labels, all_preds, target_names=encoder.classes_)
    print("\nDetailed Classification Report:")
    print(report)
    
    # 2. Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=encoder.classes_, 
                yticklabels=encoder.classes_)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Emotion Confusion Matrix')
    plt.show()

# Run the evaluation on your Test Set
evaluate_model(model, test_loader, le)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import copy

# ==========================================
# 2. Load processor (replaces your MFCC pipeline entirely)
# ==========================================
MODEL_NAME = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

def load_audio(file_path, target_sr=16000):
    """wav2vec2 expects 16kHz mono float32 waveforms."""
    try:
        audio, sr = librosa.load(file_path, sr=target_sr, mono=True, offset=0.5)
        return audio
    except:
        return None

# Load all raw waveforms
waveforms, labels = [], []
for _, row in metadata.iterrows():
    audio = load_audio(audio_dir / row['file'])
    if audio is not None:
        waveforms.append(audio)
        labels.append(row['emotion'])

le = LabelEncoder()
y = le.fit_transform(labels)
num_classes = len(le.classes_)
print(f"Classes: {list(le.classes_)}")

# ==========================================
# 3. Dataset — pads/truncates at collation time
# ==========================================
MAX_DURATION_SEC = 5
MAX_SAMPLES = 16000 * MAX_DURATION_SEC

class EmotionDataset(Dataset):
    def __init__(self, waveforms, labels):
        self.waveforms = waveforms
        self.labels = labels

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, idx):
        audio = self.waveforms[idx]
        # Truncate or pad to MAX_SAMPLES
        if len(audio) > MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
        return audio, self.labels[idx]

def collate_fn(batch):
    audios, labels = zip(*batch)
    # Processor handles normalisation (zero-mean, unit-variance)
    inputs = processor(
        list(audios),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    return inputs.input_values.to(device), torch.LongTensor(labels).to(device)

# ==========================================
# 4. Splits
# ==========================================
X_train, X_temp, y_train, y_temp = train_test_split(
    waveforms, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

def make_loader(X, y, shuffle=True, batch_size=8):
    ds = EmotionDataset(X, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

# Smaller batch size — wav2vec2 is memory-heavy
train_loader = make_loader(X_train, y_train, batch_size=8)
val_loader   = make_loader(X_val,   y_val,   shuffle=False, batch_size=8)
test_loader  = make_loader(X_test,  y_test,  shuffle=False, batch_size=8)

# ==========================================
# 5. Model
# ==========================================
class Wav2Vec2EmotionClassifier(nn.Module):
    def __init__(self, num_classes, model_name=MODEL_NAME):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)

        # Phase 1: freeze the entire encoder to train only the head first
        self.freeze_encoder()

        hidden_size = self.wav2vec2.config.hidden_size  # 768 for wav2vec2-base

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def freeze_encoder(self):
        for param in self.wav2vec2.parameters():
            param.requires_grad = False

    def unfreeze_encoder(self):
        for param in self.wav2vec2.parameters():
            param.requires_grad = True

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        # Mean-pool over the time dimension
        hidden_states = outputs.last_hidden_state  # (B, T, 768)
        pooled = hidden_states.mean(dim=1)          # (B, 768)
        return self.classifier(pooled)

model = Wav2Vec2EmotionClassifier(num_classes).to(device)

# ==========================================
# 6. Loss with class weights
# ==========================================
counts = np.bincount(y)
class_weights = torch.tensor(len(y) / (num_classes * counts), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# ==========================================
# 7. Training — two-phase
# ==========================================
def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for inputs, targets in loader:
            if train:
                optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Helps stability
                optimizer.step()

            total_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (pred == targets).sum().item()

    return total_loss / len(loader), 100 * correct / total

best_val_acc = 0.0
best_model_wts = None

# --- Phase 1: head only (fast, ~5 epochs) ---
print("Phase 1: training classifier head only...")
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

for epoch in range(5):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    _, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
    print(f"  Epoch {epoch+1}/5 | Train {train_acc:.1f}% | Val {val_acc:.1f}% | Best {best_val_acc:.1f}%")

# --- Phase 2: full fine-tune (lower LR) ---
print("\nPhase 2: full fine-tuning...")
model.unfreeze_encoder()
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

for epoch in range(20):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    _, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/20 | Train {train_acc:.1f}% | Val {val_acc:.1f}% | Best {best_val_acc:.1f}%")

# ==========================================
# 8. Test
# ==========================================
model.load_state_dict(best_model_wts)
_, test_acc = run_epoch(test_loader, train=False)
print(f"\nFinal Test Accuracy: {test_acc:.2f}%")

c:\Users\zhaoh\Desktop\CDS\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 